## Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import json
import re
import os

## Create Project Folder Structure

In [ ]:
project_path = "/content/drive/MyDrive/Fatigue-Life-ML" # define project path

folders = [
    "data/raw/Weld-Fatigue-Database-json",
    "data/processed",
    "data/archive",
    "notebooks",
    "src"
]

for folder in folders:
    os.makedirs(os.path.join(project_path, folder), exist_ok=True)

print("Project folder structure created.")

Project folder structure created.


## Define Dataset Paths

In [ ]:
param_path = project_path + "/data/raw/Weld-Fatigue-Database-json/parameter.json"
sn_path = project_path + "/data/raw/Weld-Fatigue-Database-json/S-N.json"

## Load Parameter Dataset

In [ ]:
with open(param_path, "r") as f:
    parameter_json = json.load(f)

parameters = pd.json_normalize(parameter_json["data"])

print("Parameter dataset shape:", parameters.shape)
parameters.head()

Parameter dataset shape: (5797, 38)


,dataset_id,title,doi,authors,publication,year_of_publication,UT,base_material,base_young_modulus,base_yield_strength,...,fatigue_test_type,fatigue_temp,fatigue_env,load_ratio,frequency,fatigue_machine,fatigue_standard,fatigue_test_control,stress_concentration,failure_location
0,1,Numerical fatigue assessment of welded and HFM...,10.1016/j.advengsoft.2016.01.022,"Leitner, Martin; Simunek, David; Shah, Syed Fa...",ADVANCES IN ENGINEERING SOFTWARE,2018,WOS:000432714600012,S355 mild steel,-,355,...,Axial,-,-,0.1,-,-,-,Stress and Strain-based approaches,-,Weld toe
1,2,Numerical fatigue assessment of welded and HFM...,10.1016/j.advengsoft.2016.01.022,"Leitner, Martin; Simunek, David; Shah, Syed Fa...",ADVANCES IN ENGINEERING SOFTWARE,2018,WOS:000432714600012,S355 mild steel,-,355,...,Axial,-,-,0.1,-,-,-,Stress and Strain-based approaches,-,Weld toe
2,3,Residual Stress Release and Its Effects on the...,10.1016/j.apor.2021.102673,"Li, Liangbi; Zhang, Jingxi; Zhang, Yiwen; Zhu,...",APPLIED OCEAN RESEARCH,2021,WOS:000652621000001,Modified HY130 high-strength steel,196,890,...,Axial,25,-,-,-,MTS Fatigue Testing System,-,Load,-,-
3,4,Characteristics of microstructure and fatigue ...,10.1016/j.apsusc.2013.12.157,"Yan, Shaohua; Nie, Yuan; Zhu, Zongtao; Chen, H...",APPLIED SURFACE SCIENCE,2014,WOS:000332396600002,AA5083-H111,-,-,...,Tension-Tension cyclic loading,-,-,0.1,100-110,High-frequency fatigue machine by Xian Letry C...,-,Stress,-,-
4,5,Characteristics of microstructure and fatigue ...,10.1016/j.apsusc.2013.12.157,"Yan, Shaohua; Nie, Yuan; Zhu, Zongtao; Chen, H...",APPLIED SURFACE SCIENCE,2014,WOS:000332396600002,AA5083-H111,-,-,...,Tension-Tension cyclic loading,-,-,0.1,100-110,High-frequency fatigue machine by Xian Letry C...,-,Stress,-,-


## Load Fatigue Dataset

In [ ]:
with open(sn_path, "r") as f:
    sn_json = json.load(f)

sn_data = pd.json_normalize(sn_json["data"])

print("Fatigue dataset shape:", sn_data.shape)
sn_data.head()

Fatigue dataset shape: (45315, 4)


,dataset_id,life_n,stress_range,runout
0,1,2000000.0,185.0,1
1,2,2000000.0,220.0,1
2,3,36400.0,89.0,0
3,3,31363.0,89.0,0
4,3,23461.0,89.0,0


## Merge Datasets

In [ ]:
df = pd.merge(parameters, sn_data, on="dataset_id")

print("Merged dataset shape:", df.shape)

Merged dataset shape: (45315, 41)


## Select Important Features

In [ ]:
selected_columns = [
    "base_yield_strength",
    "base_ultimate_strength",
    "welding_joint",
    "welding_method",
    "welding_voltage",
    "welding_current",
    "welding_speed",
    "fatigue_specimen_type",
    "fatigue_specimen_thickness",
    "residual_stress",
    "stress_concentration",
    "load_ratio",
    "stress_range",
    "life_n"
]

df = df[selected_columns]

## Rename Target Variable

In [ ]:
df = df.rename(columns={"life_n": "fatigue_life"})

## Function to Extract Numeric Values

In [ ]:
def extract_number(x):
    if pd.isnull(x):
        return np.nan
    x = str(x)
    numbers = re.findall(r"\d+\.?\d*", x)
    return float(numbers[0]) if numbers else np.nan

## Clean Numeric Columns

In [ ]:
numeric_columns = [
    "base_yield_strength",
    "base_ultimate_strength",
    "welding_voltage",
    "welding_current",
    "welding_speed",
    "fatigue_specimen_thickness",
    "residual_stress",
    "stress_concentration",
    "load_ratio",
    "stress_range"
]

for col in numeric_columns:
    if col in df.columns:
        df[col] = df[col].apply(extract_number)

## Replace Invalid Symbols

In [ ]:
df = df.replace("-", np.nan)

## Handle Missing Values

### Numeric Column

In [ ]:
numeric_cols = df.select_dtypes(include=["float64","int64"]).columns
df[numeric_cols] = df[numeric_cols].fillna(df[numeric_cols].median())

### Categorical columns

In [ ]:
categorical_cols = [
    "welding_joint",
    "welding_method",
    "fatigue_specimen_type"
]

df[categorical_cols] = df[categorical_cols].fillna("Unknown")

## Remove Duplicate Records

In [ ]:
duplicates = df.duplicated().sum()

print("Duplicate rows:", duplicates)

df = df.drop_duplicates()

Duplicate rows: 191


## Feature Engineering

### Stress Amplitude

In [ ]:
df["stress_amplitude"] = df["stress_range"] / 2

### Mean Stress

In [ ]:
df["mean_stress"] = df["stress_range"] * (1 + df["load_ratio"]) / 2

### Normalized Stress

In [ ]:
df["normalized_stress"] = df["stress_range"] / df["base_yield_strength"]

### Stress-Strength Ratio

In [ ]:
df["stress_strength_ratio"] = df["stress_range"] / df["base_ultimate_strength"]

## Target Transformation

In [ ]:
df["log_fatigue_life"] = np.log10(df["fatigue_life"])

## Verify Dataset

In [ ]:
print("Dataset shape:")
print(df.shape)

print("\nMissing values:")
print(df.isnull().sum())

Dataset shape:
(45124, 19)

Missing values:
base_yield_strength           0
base_ultimate_strength        0
welding_joint                 0
welding_method                0
welding_voltage               0
welding_current               0
welding_speed                 0
fatigue_specimen_type         0
fatigue_specimen_thickness    0
residual_stress               0
stress_concentration          0
load_ratio                    0
stress_range                  0
fatigue_life                  0
stress_amplitude              0
mean_stress                   0
normalized_stress             0
stress_strength_ratio         0
log_fatigue_life              0
dtype: int64


## Adding Log Fatigue Life

In [ ]:
df["log_fatigue_life"] = np.log10(df["fatigue_life"])

## Dataset Summary

In [ ]:
df.describe()

,base_yield_strength,base_ultimate_strength,welding_voltage,welding_current,welding_speed,fatigue_specimen_thickness,residual_stress,stress_concentration,load_ratio,stress_range,fatigue_life,stress_amplitude,mean_stress,normalized_stress,stress_strength_ratio,log_fatigue_life
count,45124.000000,45124.000000,45124.000000,45124.000000,45124.000000,45124.000000,45124.000000,45124.000000,45124.000000,4.512400e+04,4.512400e+04,4.512400e+04,4.512400e+04,45124.000000,45124.000000,45124.000000
mean,466.457018,580.317332,1111.659126,277.427227,11.710174,10.485675,204.296731,1.906446,0.272439,1.140479e+03,1.950572e+07,5.702393e+02,6.589859e+02,2.821699,2.514022,5.570311
std,228.213107,235.388930,11742.799588,1079.210044,72.296878,47.502141,69.180131,2.462438,0.478158,5.946771e+04,1.098865e+09,2.973385e+04,3.270723e+04,149.226238,111.070152,0.924593
min,17.060000,1.000000,2.000000,0.000000,0.010000,0.050000,0.000000,0.940000,0.000000,1.000000e+00,2.000000e+00,5.000000e-01,5.500000e-01,0.002941,0.002381,0.301030
25%,355.000000,506.000000,24.000000,185.000000,5.500000,4.000000,200.000000,1.780000,0.100000,1.280000e+02,1.160000e+05,6.400000e+01,7.500000e+01,0.312500,0.251899,5.064458
50%,400.000000,540.000000,24.000000,185.000000,5.500000,8.000000,200.000000,1.780000,0.100000,2.130000e+02,3.956810e+05,1.065000e+02,1.249500e+02,0.503650,0.398148,5.597345
75%,485.000000,579.000000,24.000000,185.000000,5.500000,12.000000,200.000000,1.780000,0.100000,3.400000e+02,1.284128e+06,1.700000e+02,2.101000e+02,0.791857,0.596296,6.108608
max,2024.000000,3724.000000,150000.000000,24100.000000,3000.000000,2024.000000,2391.000000,108.900000,10.000000,1.000000e+07,1.990000e+11,5.000000e+06,5.500000e+06,25000.000000,18518.518519,11.298853


## Save Clean Dataset

In [ ]:
output_path = project_path + "/data/processed/fatigue_dataset_clean.csv"

df.to_csv(output_path, index=False)

print("Clean dataset saved successfully")

Clean dataset saved successfully
